## Bootstrapping ratios of signficants in different variant types (common, rare, ultra-rare, singleton) cell types 

In [9]:
import pandas as pd 
import numpy as np
import os
import yaml

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf

# config 
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

#### Max script bootstrap_correltaion.py

In [ ]:
import click
import pandas as pd
import numpy as np

# options
@click.command()
@click.option('--input',
              'input_file',
              required=True,
              type=click.Path(exists=True, readable=True),
              help='TSV file')
@click.option('--group',
              'group_by',
              required=True,
              multiple=True,
              type=str,
              help='Group by column(s).')
@click.option('--iterations',
              'iterations',
              required=True,
              type=int,
              help='Number of iterations.')
@click.option('--sample-fraction',
                'sample_fraction',
                required=True,
                type=float,
                help='Sample fraction.')
@click.option('--value',
                'value',
                required=True,
                type=str,
                help='Value column.')
@click.option('--prediction',
                'predictions',
                required=True,
                multiple=True,
                type=(str, click.Choice(['abs', 'identity'])),
                help='Prediction column(s).')
@click.option('--output',
              'output_file',
              required=True,
              type=click.Path(writable=True),
              help='Final correlations.')
def cli(input_file, group_by, iterations, sample_fraction, value, predictions, output_file):

    df = pd.read_table(input_file, sep='\t').dropna()

    df_output = pd.DataFrame()
    for i in range(iterations):
        # sampling
        df_sample = df[list(group_by) + [value] + list([i[0] for i in predictions])].groupby(by=list(group_by)).sample(frac=sample_fraction, replace=False)

        # abs if needed
        df_sample_abs = df_sample.copy()
        df_sample_abs[value] = df_sample_abs[value].abs()

        # group by element
        df_sample = df_sample.groupby(by=list(group_by))
        df_sample_abs = df_sample_abs.groupby(by=list(group_by))

        df_count = df_sample.count()
        df_total_count = df.groupby(by=list(group_by)).count()

        for method in ["pearson", "spearman"]:
            df_corr= df_sample.corr(numeric_only=True, method=method)
            df_corr_abs= df_sample_abs.corr(numeric_only=True, method=method)
            for prediction, transformation in predictions:
                for element in df_corr.index:
                    if "value" in element:
                        correlation = df_corr.loc[element, prediction] if transformation == "identity" else df_corr_abs.loc[element, prediction]
                        n = df_count.loc[element[0], prediction]
                        total_n = df_total_count.loc[element[0], prediction]
                        df_output = pd.concat(
                            [
                                df_output,
                                pd.DataFrame(data=[[element[0], prediction, i, method, n, total_n, correlation]])])
    df_output.rename({0: "element", 1: "prediction", 2: "iteration", 3: "method", 4: "n", 5: "total_n", 6: "correlation"}, axis=1, inplace=True)


    # df_output = df_output.reset_index()
    df_weighted_mean = df_output.groupby(by=["iteration", "prediction", "method"]).apply(lambda x: np.average(x.correlation, weights=x.n)).reset_index()
    df_weighted_mean.rename({0: "correlation"}, axis=1, inplace=True)
    df_mean = df_output.groupby(by=["iteration", "prediction", "method"])["correlation"].mean().reset_index()
    df_weighted_mean["element"] = "Element weighted average"
    df_mean["element"] = "Element average"

    df_output = pd.concat([df_output, df_mean, df_weighted_mean], ignore_index=True)
    df_output.reset_index(inplace=True,drop=True)

    df_output.to_csv(output_file, sep="\t", index=False)

if __name__ == '__main__':
  cli()

#### Changes functionality:  (Max is kindly doing this)
- Results of his work are in `/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/IGVF_Y1_design/projects/80K_combined_variant_prediction_results/bcMPRAlm_variant_ratio.bootstrap_1000_0.8.tsv.gz`
- sample data and compute ratios with pandas as in the barplot plotting
- in throughtful_way_MPRAlm annotation: ratio calculation for each gene set in one dataframe 
- Plan:
  - for 1000 times sampling of 80% of  the initial df: 
    - compute proportion and write gene_set\tproportion to a file
    - use this file to plot x=gene_set and y=proportion + ci

In [10]:
# read it in 
bootstrapped_variants_path = config['files']['creating']['bootstrabed_bcMPRAlm_results']
bootstrapped_variants = pd.read_csv(bootstrapped_variants_path, sep="\t")
bootstrapped_variants

,gene_set,variant_type,iteration,n,total_n,ratio
0,cardiac,common,0,2563,3201,0.010535
1,cardiac,rare,0,2036,2551,0.008841
2,cardiac,singleton,0,2235,2793,0.017450
3,cardiac,ultra-rare,0,2221,2774,0.014858
4,cava,common,0,579,712,0.012090
...,...,...,...,...,...,...
15995,neuro,ultra-rare,999,2335,2873,0.021413
15996,random,common,999,242,293,0.012397
15997,random,rare,999,194,241,0.020619
15998,random,singleton,999,1176,1475,0.013605


In [ ]:
# plot ratios with confidence intervall

##### not used/modified yet

In [ ]:
# df = pd.read_table(input_file, sep='\t').dropna()

#     df_output = pd.DataFrame()
#     for i in range(iterations):
#         # sampling
#         df_sample = df[list(group_by) + [value] + list([i[0] for i in predictions])].groupby(by=list(group_by)).sample(frac=sample_fraction, replace=False)

#         # abs if needed
#         df_sample_abs = df_sample.copy()
#         df_sample_abs[value] = df_sample_abs[value].abs()

#         # group by element
#         df_sample = df_sample.groupby(by=list(group_by))
#         df_sample_abs = df_sample_abs.groupby(by=list(group_by))

#         df_count = df_sample.count()
#         df_total_count = df.groupby(by=list(group_by)).count()

#         for method in ["pearson", "spearman"]:
#             df_corr= df_sample.corr(numeric_only=True, method=method)
#             df_corr_abs= df_sample_abs.corr(numeric_only=True, method=method)
#             for prediction, transformation in predictions:
#                 for element in df_corr.index:
#                     if "value" in element:
#                         correlation = df_corr.loc[element, prediction] if transformation == "identity" else df_corr_abs.loc[element, prediction]
#                         n = df_count.loc[element[0], prediction]
#                         total_n = df_total_count.loc[element[0], prediction]
#                         df_output = pd.concat(
#                             [
#                                 df_output,
#                                 pd.DataFrame(data=[[element[0], prediction, i, method, n, total_n, correlation]])])
#     df_output.rename({0: "element", 1: "prediction", 2: "iteration", 3: "method", 4: "n", 5: "total_n", 6: "correlation"}, axis=1, inplace=True)


#     # df_output = df_output.reset_index()
#     df_weighted_mean = df_output.groupby(by=["iteration", "prediction", "method"]).apply(lambda x: np.average(x.correlation, weights=x.n)).reset_index()
#     df_weighted_mean.rename({0: "correlation"}, axis=1, inplace=True)
#     df_mean = df_output.groupby(by=["iteration", "prediction", "method"])["correlation"].mean().reset_index()
#     df_weighted_mean["element"] = "Element weighted average"
#     df_mean["element"] = "Element average"

#     df_output = pd.concat([df_output, df_mean, df_weighted_mean], ignore_index=True)
#     df_output.reset_index(inplace=True,drop=True)

#     df_output.to_csv(output_file, sep="\t", index=False)